In [1]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
WRITE = False

In [3]:
data = json.load(open(os.path.join('tts_eval', 'rankings.json'), 'r', encoding='utf-8'))

In [5]:
elos = pd.read_csv(os.path.join('tts_eval', 'elo_rankings.csv'))

In [12]:
wins = dict.fromkeys(('valle', 'parler', 'gtts', 'mms', 'bark', 'piper'), 0)
losses = dict.fromkeys(('valle', 'parler', 'gtts', 'mms', 'bark', 'piper'), 0)

In [13]:
for key in data:
    results = data[key]
    for pair in results:
        wins[pair[0]] += 1
        losses[pair[1]] += 1

In [14]:
wins

{'valle': 33, 'parler': 39, 'gtts': 42, 'mms': 6, 'bark': 62, 'piper': 43}

In [15]:
losses

{'valle': 42, 'parler': 36, 'gtts': 33, 'mms': 69, 'bark': 13, 'piper': 32}

In [16]:
rates = pd.DataFrame({
    'wins': wins,
    'losses': losses
})

In [17]:
rates

,wins,losses
valle,33,42
parler,39,36
gtts,42,33
mms,6,69
bark,62,13
piper,43,32


In [18]:
rates['win_rate'] = rates['wins'] / (rates['wins'] + rates['losses'])

In [22]:
rates = rates.sort_values(by=['win_rate'], ascending=False)

In [23]:
rates

,wins,losses,win_rate
bark,62,13,0.826667
piper,43,32,0.573333
gtts,42,33,0.560000
parler,39,36,0.520000
valle,33,42,0.440000
mms,6,69,0.080000


In [25]:
print(rates.to_latex())

\begin{tabular}{lrrr}
\toprule
 & wins & losses & win_rate \\
\midrule
bark & 62 & 13 & 0.826667 \\
piper & 43 & 32 & 0.573333 \\
gtts & 42 & 33 & 0.560000 \\
parler & 39 & 36 & 0.520000 \\
valle & 33 & 42 & 0.440000 \\
mms & 6 & 69 & 0.080000 \\
\bottomrule
\end{tabular}



In [27]:
latencies = dict.fromkeys(wins.keys())

In [28]:
latency_df = pd.DataFrame(columns=range(15), index=latencies.keys())

In [45]:
for key in latencies:
    df = pd.read_csv(os.path.join('tts_eval', key, 'latencies.csv'), header=None)
    for idx, row in df.iterrows():
        prompt_id = int(row[0])
        latency = float(row[1])
        latency_df.loc[key, prompt_id] = latency

In [53]:
for col in latency_df.columns:
    latency_df[col] = latency_df[col].astype(float)

In [47]:
latency_df = latency_df.transpose()

In [55]:
described = latency_df.describe()

In [57]:
described = described.transpose()

In [59]:
described['index_copy'] = described.index

In [64]:
described = described.sort_values(by=['mean'])

In [67]:
fig = px.bar(
    described,
    x = 'index_copy',
    y = 'mean',
    title = 'Mean latency by model in seconds (↓)',
    width = 600,
    height = 400
)
# fig.update_traces(marker=dict(size=8))
fig.update_layout(
    xaxis = dict(
        tickmode = 'array',
        tickvals = described['index_copy']
    )
)
if WRITE:
    fig.write_image(os.path.join('figs', 'latency_bars.pdf'), format='pdf')
fig.show()